In [44]:
import rasterio
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from dist_s1.dist_plot import get_dist_s1_mpl_cmap
from tqdm import tqdm
from dem_stitcher.rio_window import read_raster_from_window

In [67]:
MGRS_TILE_ID = '20LNJ' #'22MGB' #'33MXS' #'50NPL' #'21MWP' #'19LCF' # '20LMM' '20MMA'
MGRS_TILE_ID = '21MWP'
MGRS_TILE_ID = '19LCF'
MGRS_TILE_ID = '50NPL'

In [68]:
if MGRS_TILE_ID in ['20LNJ']:
    bounds = None
elif MGRS_TILE_ID in ['19LCF']:
    bounds = (-70.216, -12.9612, -70.123,-12.912)
elif MGRS_TILE_ID in ['21MWP']:
    bounds = (-56.7526, -5.9644, -56.52545, -5.8433)
elif MGRS_TILE_ID in ['50NPL']:
    bounds = (118.0082, 5.084, 118.5693, 5.4602)
elif MGRS_TILE_ID in ['33MXS']:
    bounds = (16.24222, -2.832366, 16.29601, -2.7855)
elif MGRS_TILE_ID in ['22MGB']:
    bounds = (-49.0518, -3.3957, -48.3151, -2.7542)
else:
    raise ValueError(f'MGRS_TILE_ID {MGRS_TILE_ID} not supported')


In [69]:
COMP_DIR = Path(f'comparison_dir / {MGRS_TILE_ID}')
COMP_DIR.exists()

True

In [70]:
radd_path = list(COMP_DIR.glob('radd*alerts.tif'))[0]
dist_s1_path = COMP_DIR / f'{MGRS_TILE_ID}_status_agg_all.tif'
dist_hls_gen_path = list(COMP_DIR.glob(f'{MGRS_TILE_ID}*gen*.tif'))[0]
dist_hls_veg_path = list(COMP_DIR.glob(f'{MGRS_TILE_ID}*veg*.tif'))[0]
radd_path.exists(), dist_s1_path.exists(), dist_hls_gen_path.exists(), dist_hls_veg_path.exists()


(True, True, True, True)

In [71]:
if bounds is None:
    with rasterio.open(radd_path) as src:
        radd_data = src.read(1)

    with rasterio.open(dist_s1_path) as src:
        dist_s1_data = src.read(1)

    with rasterio.open(dist_hls_gen_path) as src:
        dist_hls_gen_data = src.read(1)

    with rasterio.open(dist_hls_veg_path) as src:
        dist_hls_veg_data = src.read(1)

else:
    radd_data, p_radd = read_raster_from_window(radd_path, bounds)
    radd_data = radd_data[0, ...]
    dist_s1_data, p_dist_s1 = read_raster_from_window(dist_s1_path, bounds)
    dist_s1_data = dist_s1_data[0, ...]
    dist_hls_gen_data, p_dist_hls_gen = read_raster_from_window(dist_hls_gen_path, bounds)
    dist_hls_gen_data = dist_hls_gen_data[0, ...]
    dist_hls_veg_data, p_dist_hls_veg = read_raster_from_window(dist_hls_veg_path, bounds)
    dist_hls_veg_data = dist_hls_veg_data[0, ...]



/Users/cmarshak/miniforge3/envs/dist-s1-env/lib/python3.13/site-packages/dem_stitcher/rio_window.py:144: RuntimeWarning: Requesting extent beyond raster bounds of [117.90127592636325, 4.432467274902542, 118.89355498920168, 5.427620946650149]. Shrinking bounds in raster crs to (118.0082, 5.084, 118.5693, 5.427620946650149).
  warn(
/Users/cmarshak/miniforge3/envs/dist-s1-env/lib/python3.13/site-packages/dem_stitcher/rio_window.py:144: RuntimeWarning: Requesting extent beyond raster bounds of [600000.0, 490200.0, 709800.0, 600000.0]. Shrinking bounds in raster crs to (611754.7079679914, 562036.6787525218, 673858.207872475, 600000.0).
  warn(
/Users/cmarshak/miniforge3/envs/dist-s1-env/lib/python3.13/site-packages/dem_stitcher/rio_window.py:144: RuntimeWarning: Requesting extent beyond raster bounds of [600000.0, 490200.0, 709800.0, 600000.0]. Shrinking bounds in raster crs to (611754.7079679914, 562036.6787525218, 673858.207872475, 600000.0).
  warn(
/Users/cmarshak/miniforge3/envs/dist-

In [72]:
dist_ts_dir = Path('opera_dist_data/')
ts_dir = list(dist_ts_dir.glob(f'*_{MGRS_TILE_ID}/'))[0]
ts_dir

PosixPath('opera_dist_data/treelosswet__50NPL')

In [73]:
PNG_DIR = Path(f'pngs/{ts_dir.stem}')
PNG_DIR.mkdir(exist_ok=True, parents=True)


In [74]:
dist_cmap, dist_norm = get_dist_s1_mpl_cmap()

In [ ]:
from os import name


for arr, name in tqdm([(radd_data, 'radd'), (dist_s1_data, 'dist_s1'), (dist_hls_gen_data, 'dist_hls_gen'), (dist_hls_veg_data, 'dist_hls_veg')]):
    fig, ax = plt.subplots(dpi=1_000)
    ax.imshow(arr, cmap=dist_cmap, norm=dist_norm)
    ax.set_axis_off()
    out_png = PNG_DIR / f'{MGRS_TILE_ID}_{name}.png'
    plt.savefig(out_png, bbox_inches='tight', pad_inches=0)
    plt.close('all')


  0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
import contextily as cx
def get_basemap_for_matplotlib(w, s, e, n, zoom="auto", source=cx.providers.Esri.WorldImagery):
    # Google Satellite: https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}
    img, extent = cx.bounds2img(w, s, e, n, zoom=zoom, source=source, ll=True)
    return img, extent

In [ ]:
if bounds is not None:
    img, extent = get_basemap_for_matplotlib(bounds[0], bounds[1], bounds[2], bounds[3])
    fig, ax = plt.subplots(dpi=1_000)
    ax.imshow(img)
    ax.set_axis_off()
    out_png = PNG_DIR / f'{MGRS_TILE_ID}_esri_basemap.png'
    plt.savefig(out_png, bbox_inches='tight', pad_inches=0)
    plt.close('all')